<a href="https://colab.research.google.com/github/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/blob/main/my%20training%20code/step3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 強制解除安裝可能發生衝突的套件
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers

# 重新安裝穩定且互相相容的版本組合
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.40.1 sentence-transformers==2.7.0 snorkel

Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (780.4 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.20.1%2Bcu121-cp312-cp312-linux_x86_64.whl (7.3 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchaudio-2.5.1%2Bcu121-cp312-cp312-linux_x86_64.whl (3.4 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

In [2]:
!pip install -q transformers==4.40.1 sentence-transformers==2.7.0 snorkel

In [3]:
import pandas as pd
import yaml
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.model import LabelModel
from sklearn.svm import LinearSVC
import urllib.request

# ==========================================
# 1. 初始化與模型載入
# ==========================================
print("Loading model and configs...")
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device=device)

# 載入 YAML 配置
YAML_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/training%20data/keywords.yaml"
yaml_response = urllib.request.urlopen(YAML_URL)
config = yaml.safe_load(yaml_response)

# 載入人工審核資料 (Gold Dataset) 作為標註錨點
print("正在載入人工審核資料作為標註錨點...")
# 🚨 這裡已經換成你指定的最新檔案
GOLD_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/gold_candidates_review_with_ignore.csv"
df_reviewed = pd.read_csv(GOLD_URL).dropna(subset=['text'])
# 確保 proposed_category 是英文
reviewed_mapping = dict(zip(df_reviewed['text'], df_reviewed['proposed_category']))

# 定義完整的標籤映射 (0~5)
ABSTAIN = -1
NORMAL = 0
LOAN = 1
PORN = 2
GAMBLING = 3
SCAM = 4
DRUGS = 5

label_map = {
    "loan": LOAN,
    "porn": PORN,
    "gambling": GAMBLING,
    "scam_recruitment": SCAM,
    "drugs": DRUGS
}

# ==========================================
# 2. 多類別 TCAV 準備 (直接從 GitHub 讀取 txt)
# ==========================================
def compute_cav(concept_texts, random_texts, encoder):
    concept_embs = encoder.encode(concept_texts)
    random_embs = encoder.encode(random_texts)
    X = np.vstack((concept_embs, random_embs))
    y = np.array([1] * len(concept_embs) + [0] * len(random_embs))
    svm = LinearSVC(random_state=42, max_iter=2000, dual=False)
    svm.fit(X, y)
    cav = svm.coef_[0]
    return cav / np.linalg.norm(cav)

def load_txt_from_github(url):
    response = urllib.request.urlopen(url)
    return [line.decode('utf-8').strip() for line in response if line.strip()]

print("正在從 GitHub 載入 TCAV 概念資料集...")
BASE_URL = "https://raw.githubusercontent.com/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/main/concept%20data"

# 直接從 GitHub 讀取你剛剛上傳的真實抽樣對照組
random_normals = load_txt_from_github(f"{BASE_URL}/random_normal.txt")

concepts_data = {
    "loan": load_txt_from_github(f"{BASE_URL}/concept_loan.txt"),
    "porn": load_txt_from_github(f"{BASE_URL}/concept_porn.txt"),
    "gambling": load_txt_from_github(f"{BASE_URL}/concept_gambling.txt"),
    "drugs": load_txt_from_github(f"{BASE_URL}/concept_drugs.txt"),
    "scam_recruitment": load_txt_from_github(f"{BASE_URL}/concept_scam_recruitment.txt")
}

# 3. 開始計算每一把量尺 (CAV)
cav_dict = {}
for cat_name, texts in concepts_data.items():
    print(f"正在計算 {cat_name} 的 CAV (使用 {len(texts)} 筆資料)...")
    cav_dict[cat_name] = compute_cav(texts, random_normals, embedder)

# ==========================================
# 3. 定義所有的 Labeling Functions (LFs)
# ==========================================
# 🚨 刪除了 lf_negative_hit

@labeling_function(resources=dict(cfg=config))
def lf_strong_hit(x, cfg):
    text = x['name']
    for cat_id, cat_info in cfg['categories'].items():
        if cat_id in label_map:
            rules = cat_info.get('fine_grained_rules', {})
            strong_words = rules.get('strong_hits', [])
            if any(w in text for w in strong_words):
                return label_map.get(cat_id, ABSTAIN)
    return ABSTAIN

@labeling_function(resources=dict(cfg=config))
def lf_weak_hit_with_context(x, cfg):
    text = x['name']
    for cat_id, cat_info in cfg['categories'].items():
        if cat_id in label_map:
            rules = cat_info.get('fine_grained_rules', {})
            logic = rules.get('weak_hit_logic', {})
            if not logic: continue
            targets = logic.get('targets', [])
            triggers = logic.get('triggers', [])
            if any(t in text for t in targets) and any(tr in text for tr in triggers):
                return label_map.get(cat_id, ABSTAIN)
    return ABSTAIN

@labeling_function()
def lf_tcav_multiclass(x):
    text_emb = embedder.encode(x['name'])

    scores = {
        LOAN: np.dot(text_emb, cav_dict["loan"]),
        PORN: np.dot(text_emb, cav_dict["porn"]),
        GAMBLING: np.dot(text_emb, cav_dict["gambling"]),
        DRUGS: np.dot(text_emb, cav_dict["drugs"]),
        SCAM: np.dot(text_emb, cav_dict["scam_recruitment"])
    }

    best_class = max(scores, key=scores.get)
    highest_score = scores[best_class]

    # 門檻設定，可視結果微調 (例如調高到 0.6)
    threshold = 0.5
    if highest_score > threshold:
        return best_class
    return ABSTAIN

@labeling_function(resources=dict(mapping=reviewed_mapping))
def lf_ground_truth(x, mapping):
    text = x['name']
    if text in mapping:
        label_name = mapping[text]
        return label_map.get(label_name, ABSTAIN)
    return ABSTAIN

# ==========================================
# 4. 執行標註與訓練 LabelModel
# ==========================================
lfs = [lf_strong_hit, lf_weak_hit_with_context, lf_tcav_multiclass, lf_ground_truth]

print("\n載入待標註資料 (使用最新生成的候選池)...")
# 🚨 這裡直接讀取你在 Step 1 產生、Step 2 脫敏後的大量候選資料
CANDIDATE_URL = "https://media.githubusercontent.com/media/aaallyssaaa/Deconstructive-analysis-of-malicious-semantics/refs/heads/main/training%20data/candidate_pool_sanitized.csv"
df_train = pd.read_csv(CANDIDATE_URL)
df_train['name'] = df_train['name'].fillna("").astype(str)

print("\n--- 開始執行標註 (Applier) ---")
applier = PandasLFApplier(lfs=lfs)
L_train = applier.apply(df=df_train)

print("\n--- 標註函數效能分析 ---")
display(LFAnalysis(L=L_train, lfs=lfs).lf_summary())

print("\n訓練 Snorkel LabelModel 中...")
# 🚨 類別總共有 6 種 (0~5)，所以 cardinality = 6
label_model = LabelModel(cardinality=6, verbose=True)
label_model.fit(L_train=L_train, n_epochs=500, log_freq=100, seed=42)

df_train['snorkel_label_idx'] = label_model.predict(L=L_train, tie_break_policy="abstain")
df_train['snorkel_label'] = df_train['snorkel_label_idx'].map({
    NORMAL: "正常困難負樣本",
    LOAN: "借貸融資",
    PORN: "黃色與特種行業",
    GAMBLING: "博弈",
    SCAM: "詐騙高風險招募",
    DRUGS: "毒品",
    ABSTAIN: "棄權未分類"
})

df_labeled = df_train[df_train['snorkel_label_idx'] != ABSTAIN].copy()
df_labeled.to_csv('snorkel_labeled_training_data.csv', index=False, encoding='utf-8-sig')

print(f"\n✅ 標註完成！最終有效樣本數: {len(df_labeled)}")
print(df_labeled['snorkel_label'].value_counts())

Loading model and configs...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...


--- 標註函數效能分析 ---


,j,Polarity,Coverage,Overlaps,Conflicts
lf_strong_hit,0,"[1, 2, 4]",0.344839,0.092426,0.007525
lf_weak_hit_with_context,1,"[2, 4]",0.007852,0.001636,0.000164
lf_tcav_multiclass,2,"[1, 2, 3, 4, 5]",0.091117,0.078194,0.007361
lf_ground_truth,3,"[1, 2, 3, 4]",0.078848,0.022738,0.000327



訓練 Snorkel LabelModel 中...


100%|██████████| 500/500 [00:00<00:00, 960.20epoch/s]


✅ 標註完成！最終有效樣本數: 2581
snorkel_label
借貸融資       1922
黃色與特種行業     395
博弈          192
詐騙高風險招募      70
毒品            2
Name: count, dtype: int64
